In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools
from sklearn.metrics.cluster import adjusted_mutual_info_score, adjusted_rand_score
from scipy import stats
from scipy.stats import pearsonr, kruskal, chi2_contingency
import seaborn as sns
from ptitprince import PtitPrince as pt
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from pandas.api.types import is_numeric_dtype
import statsmodels
from statsmodels.stats.multitest import multipletests

In [ ]:
# Function to remove clusters containing < 10% patients
# output is also list with top x outlier patients (patients most typically found in small clusters)
def remove_small_clusters(df: pd.DataFrame, n_outliers: int, verbose: bool):
    valid_results = df[df['relative_cluster_sizes'].apply(lambda d: all(value >= 0.1 for value in d.values()))]
    outlier_results = df[df['relative_cluster_sizes'].apply(lambda f: any(value < 0.1 for value in f.values()))]
    outlier_patients = {}
    for index, row in outlier_results.iterrows():
        cluster_sizes = row['relative_cluster_sizes']
        small_clusters = [cluster for cluster, size in cluster_sizes.items() if size < 0.1]
        patients = row['y_pred_idx']
        clusters = row['y_pred']
        for patient, cluster in zip(patients, clusters):
            if cluster in small_clusters:
                if patient not in outlier_patients:
                    outlier_patients[patient] = 1
                else:
                    outlier_patients[patient] += 1
    for key in outlier_patients:
        outlier_patients[key] /= len(outlier_results)
    top_outliers = sorted(outlier_patients.items(), key=lambda x: x[1], reverse=True)[:n_outliers]
    df_top_outliers = pd.DataFrame(data=top_outliers, columns=['Patient ID', 'Count'])
    top_outlier_patients = [patient for patient, count in top_outliers]
    if verbose == True:
        print(f"Top {n_outliers} outlier patients: {top_outlier_patients}")
    return valid_results, outlier_results, df_top_outliers

In [ ]:
# Function to perform clinical enrichment of labels to each row
def clinical_enrichment(results, clinical_data):
    # Sort labels
    results["sorted_y_pred_idx"] = results["y_pred_idx"].apply(sorted)
    results["sorted_y_pred"] = results.apply(
        lambda row: [row["y_pred"][row["y_pred_idx"].index(patient_id)] for patient_id in row["sorted_y_pred_idx"]],
        axis=1)
    
    # Add clinical data
    clinical_data = clinical_data[['Patient ID', 'Fraction Genome Altered', 'Diagnosis Age', 'Sex', 'Race Category', 
                                   'Adjuvant Postoperative Targeted Therapy Administered Indicator', 'Alcohol History Documented', 'Tumor resected max dimension',
                                   'American Joint Committee on Cancer Metastasis Stage Code', 'American Joint Committee on Cancer Tumor Stage Code',
                                   'Chronic Pancreatitis Personal Medical History Indicator', 'Did patient start adjuvant postoperative radiotherapy?', 
                                   'Disease Free Status', 'Family History of Cancer', 'Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code', 
                                   'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Neoplasm Histologic Grade', 'TMB (nonsynonymous)',
                                   'New Neoplasm Event Post Initial Therapy Indicator', 'Overall Survival (Months)', 'Overall Survival Status', 'Disease Free (Months)', 
                                   'Participant Personal Medical History Diabetes Mellitus Ind-3', 'Patient Primary Tumor Site', 'Prior Cancer Diagnosis Occurence', 
                                   'Surgical Margin Resection Status', 'Patient Smoking History Category', 'Person Neoplasm Status', 'Primary Therapy Outcome Success Type']]
    clinical_data.set_index('Patient ID', inplace=True)
    
    # Convert necessary data 
    clinical_data['Overall Survival Status'] = clinical_data['Overall Survival Status'].str.split(':').str[0].astype(int)
    clinical_data['Patient Smoking History Category'] = (clinical_data['Patient Smoking History Category']
                                                         .where(clinical_data['Patient Smoking History Category'].isna(), 
                                                                clinical_data['Patient Smoking History Category'].astype(float).astype(str)))
    clinical_data_columns = [col for col in clinical_data.columns if col != 'Patient ID']

    def get_filtered_clinical_data(patient_ids, clinical_data, column_name):
        filtered_data = clinical_data.loc[patient_ids]
        return filtered_data[column_name].values.tolist()
    for column in clinical_data_columns:
        results[column] = results.apply(lambda row: get_filtered_clinical_data(row['sorted_y_pred_idx'], clinical_data, column), axis=1)
    
    # Logrank test
    def calculate_logrank_pvalue(row):
        df = pd.DataFrame({
            'cluster': row['sorted_y_pred'],
            'vital_status': row['Overall Survival Status'],
            'days_to_death': row['Overall Survival (Months)']
        })
        kmf = KaplanMeierFitter()
        test_results = multivariate_logrank_test(df['days_to_death'], df['cluster'], df['vital_status'])
        return test_results.p_value
    results['pvalue_logrank'] = results.apply(calculate_logrank_pvalue, axis=1)
    
    # Function for p-values (Kruskal-Wallis, chi2)
    def pvalue_tests(row, clinical_data_columns):
        pvalues = []
        for variable in clinical_data_columns:
            df = pd.DataFrame({
                'cluster': row['y_pred'],
                variable: row[variable]
            })
            if pd.api.types.is_numeric_dtype(df[variable]):
                # Kruskal-Wallis test for numerical variables
                test_numerical = [df[df['cluster'] == cluster][variable].dropna().to_numpy() for cluster in df['cluster'].unique()]
                stat, p_value_kruskal = kruskal(*test_numerical)
                pvalues.append(p_value_kruskal)
            else:
                # Chi-square contingency test for categorical variables
                test_discrete = pd.crosstab(df['cluster'], df[variable])
                chi2, p_value_chi2, dof, freq = chi2_contingency(test_discrete)
                pvalues.append(p_value_chi2)
        reject, pvals_corr, asidack, abonf = multipletests(pvals=pvalues, alpha=0.05, method='fdr_bh')
        for idx, variable in enumerate(clinical_data_columns):
            row[f"pvalue_{variable}"] = pvals_corr[idx]
        return row
    
    clinical_data_columns = [col for col in clinical_data_columns if col not in ['Overall Survival Status', 'Overall Survival (Months)']]
    results = results.apply(lambda row: pvalue_tests(row, clinical_data_columns), axis=1)
    columns_to_check = [f'pvalue_{variable}' for variable in clinical_data_columns]
    results['n_enriched_clinical'] = (results[columns_to_check] < 0.05).sum(axis=1)
    return results

In [ ]:
# Function to calculate stability metrics
def calculate_stability_metrics(results: pd.DataFrame, random_state=None, progress_bar=True):

    alg_stability = results[['dataset', 'view_combination', 'algorithm', 'n_clusters', 'missing_percentage', 'amputation_mechanism', 'imputation', 'run_n', "sorted_y_pred", 
                             "sorted_y_pred_idx", 'silhouette', 'normalised_silhouette', 'vrc', 'db', 'dbcv', 'dunn', "dhi", "ssei", 'rsi', 'bhi', 'pvalue_logrank', 'n_enriched_clinical']]
    if alg_stability["imputation"].nunique() != 1:
        alg_stability = alg_stability.loc[
            (alg_stability["missing_percentage"] == 0) | (alg_stability["imputation"])
            ]
    alg_uns_metrics = alg_stability.drop(columns=["sorted_y_pred", "sorted_y_pred_idx",'imputation', 'run_n'])
    
    # Group by taking mean of metrics
    alg_uns_metrics = alg_uns_metrics.groupby(
        ["dataset", "algorithm", "missing_percentage", "amputation_mechanism", "view_combination", "n_clusters"], as_index=False).mean()

    iterator = alg_stability["dataset"].unique()
    if progress_bar:
        iterator = tqdm(iterator)

    for dataset in iterator:
        preds_dataset = alg_stability.loc[
            (alg_stability["dataset"] == dataset), ["missing_percentage", "algorithm", 'amputation_mechanism', 'n_clusters', 
                                                    'view_combination', "run_n", "sorted_y_pred", "sorted_y_pred_idx"]]
        for alg in preds_dataset["algorithm"].unique():
            pred_alg = preds_dataset[preds_dataset["algorithm"] == alg]
            for missing_percentage in pred_alg["missing_percentage"].unique():
                pred_missing_alg = pred_alg[pred_alg["missing_percentage"] == missing_percentage]
                for amputation_mechanism in pred_missing_alg["amputation_mechanism"].unique():
                    pred_missing_ampt_alg = pred_missing_alg[
                        pred_missing_alg["amputation_mechanism"] == amputation_mechanism]
                    
                    for view in pred_missing_ampt_alg["view_combination"].unique():
                        pred_missing_ampt_alg_view = pred_missing_ampt_alg[
                            pred_missing_ampt_alg["view_combination"] == view]
                        for cluster in pred_missing_ampt_alg_view["n_clusters"].unique():
                            pred_missing_ampt_alg_view_clus = pred_missing_ampt_alg_view[
                                pred_missing_ampt_alg_view['n_clusters'] == cluster]

                            amis, aris = [], []
                            
                            for run_1, run_2 in set(itertools.combinations(pred_missing_ampt_alg_view_clus["run_n"].unique(), 2)):
                                pred1_alg = pred_missing_ampt_alg_view_clus.loc[
                                    (pred_missing_ampt_alg_view_clus["run_n"] == run_1), "sorted_y_pred"].to_list()[0]
                                pred2_alg = pred_missing_ampt_alg_view_clus.loc[
                                    (pred_missing_ampt_alg_view_clus["run_n"] == run_2), "sorted_y_pred"].to_list()[0]
                                
                                pred1_idx = pred_missing_ampt_alg_view_clus.loc[(
                                    pred_missing_ampt_alg_view_clus["run_n"] == run_1), "sorted_y_pred_idx"].to_list()[0]
                                pred2_idx = pred_missing_ampt_alg_view_clus.loc[(
                                    pred_missing_ampt_alg_view_clus["run_n"] == run_2), "sorted_y_pred_idx"].to_list()[0]
        
                                # Only select samples in common for stability metrics
                                common_samples = list(set(pred1_idx) & set(pred2_idx))
                                pred1_common = [pred1_alg[pred1_idx.index(i)] for i in common_samples]
                                pred2_common = [pred2_alg[pred2_idx.index(i)] for i in common_samples]
        
                                amis.append(adjusted_mutual_info_score(pred1_common, pred2_common)), aris.append(
                                    adjusted_rand_score(pred1_common, pred2_common))
        
                            alg_uns_metrics.loc[(alg_uns_metrics["dataset"] == dataset) &
                                                (alg_uns_metrics["missing_percentage"] == missing_percentage) &
                                                (alg_uns_metrics["amputation_mechanism"] == amputation_mechanism) &
                                                (alg_uns_metrics["algorithm"] == alg) & 
                                                (alg_uns_metrics["view_combination"] == view) & 
                                                (alg_uns_metrics["n_clusters"] == cluster),
                            ["AMI", "ARI"]] = [np.mean(amis), np.mean(aris)]
    return alg_uns_metrics

In [ ]:
# Function to normalise metrics with respect to a variable
def add_normalised_metric(df, variable_to_normalise, metric, greater_is_better=True):
    possible_variables = ["dataset", "algorithm", "missing_percentage", "amputation_mechanism", "view_combination", "n_clusters"]
    valid_variables = [var for var in possible_variables if var != variable_to_normalise]
    def recursive_loop(subset, remaining_vars, current_filters):
        if not remaining_vars:
            scores = subset[metric].values
            relative_score = scores / scores.max()
            if not greater_is_better:
                relative_score = 1 - relative_score
            condition = True
            for key, value in current_filters.items():
                condition &= (df[key] == value)
            df.loc[condition, f'normalised_{metric}'] = relative_score
            return
        current_var = remaining_vars[0]
        for unique_value in subset[current_var].unique():
            filtered_subset = subset[subset[current_var] == unique_value]
            recursive_loop(filtered_subset, remaining_vars[1:], {**current_filters, current_var: unique_value})
    df[f'normalised_{metric}'] = float('nan')
    recursive_loop(df, valid_variables, {})
    return df

# First benchmark: obtaining the best view combination

The aim of this benchmark is to find the best combination of omic views that will give the best clusters. Six omic views from TCGA PDAC database will be used: protein expression arrays (RPPA), RNA-sequence expression (RNAseq), micro-RNA expression (miRNA), mutations binary data (mutations; 1 for at least one mutation present in gene, 0 for no mutations present), methylation, and somatic copy number variations (CNA) in GISTIC2.0 format (-2, -1, 0, 1, 2). All the possible combinations between these data types, containing at least two views were considered, resulting in $2^6 -1 -1 = 57$ combinations. 

This benchmark will also be used to analyse some hyperparameters, such as the number of clusters (2 vs. 3), the data types present, the number of data types and the algorithms used in each experiment. 

In [ ]:
results1_error = pd.read_csv('benchmarking_files/first_bench.csv', dtype={'view_combination': str})
results1_update = pd.read_csv('benchmarking_files/first_bench_line.csv', dtype={'view_combination': str})
index_to_replace = results1_error[results1_error['silhouette'].isna()].index
results1_error.loc[index_to_replace[0]] = results1_update.iloc[0]
results1_error.to_csv('benchmarking_files/first_bench_file.csv', index=False)

In [ ]:
results1_file = pd.read_csv('benchmarking_files/first_bench_file.csv',
                            dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval, 
                                        'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
valid_results1, outlier_results1, outlier_patients1 = remove_small_clusters(results1_file, 20, verbose = False)
results_clin1 = clinical_enrichment(valid_results1, clinical_data_file)

In [ ]:
normalised_silhouette1 = add_normalised_metric(results_clin1, variable_to_normalise='view_combination', metric='silhouette', greater_is_better=True)
stability_metrics_results1 = calculate_stability_metrics(normalised_silhouette1, random_state=42, progress_bar=True)
normalised_ami1 = add_normalised_metric(stability_metrics_results1, variable_to_normalise='view_combination', metric='AMI', greater_is_better=True)
results1 = normalised_ami1.copy()
results1['n_views'] = normalised_ami1['view_combination'].str.count('1')
# Creating a combined metric using silhouette score (quality of clusters) and AMI (stability metric)
results1['combined_metric'] = results1[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results1.sort_values('combined_metric', ascending=False, inplace=True)
results1

In [ ]:
metric = 'combined_metric'
views = ['CNA', 'Methyl', 'Mutation', 'RNAseq', 'RPPA', 'miRNA']

combinations_average = results1.groupby(['view_combination']).mean(numeric_only=True).sort_values(by=metric, ascending=False)
combinations_sorted = combinations_average.index.tolist()

fig, ax = plt.subplots(4, 1, sharex=True, figsize=(18,7), height_ratios=[0.5, 0.3, 0.1, 0.1])

# First plot: boxplots with combined metric score for each combination
sns.boxplot(data=results1, x='view_combination', y=metric, ax=ax[0], 
            order=combinations_sorted, width=0.7, color='white', showmeans=True)
ax[0].set_ylabel('AMI + silhouette score (normalised)')
ax[0].set_xlabel('')
ax[0].grid(alpha=0.4, which='both', axis='both')
ax[0].set_axisbelow(True)
ax[0].set_ylim(-0.1, 1.1)
for line in ax[0].lines:
    line.set_color('black')
    line.set_xdata(line.get_xdata() + 0.5)
for patch in ax[0].patches:
    patch.set_edgecolor('black')
    vertices = patch.get_path().vertices
    vertices[:, 0] += 0.5

# Second plot: heatmap showing modalities present 
views_matrix = combinations_average.reset_index()
views_matrix_expanded = views_matrix['view_combination'].apply(lambda x: pd.Series(list(x))).astype(int)
views_matrix_expanded.columns = views
views_matrix_expanded.index = views_matrix['view_combination']
views_ordered = ['CNA', 'Methyl', 'miRNA', 'RNAseq', 'RPPA', 'Mutation']
views_matrix_expanded = views_matrix_expanded.reindex(columns=views_ordered)
sns.heatmap(views_matrix_expanded.T, cmap='Blues', linewidths=0.1, linecolor='black', 
            cbar=False, ax=ax[1]).set(xlabel=None)
ax[1].set_ylabel('Modalities')
ax[1].tick_params(axis='x', bottom=True, labelbottom=False)

# Third plot: heatmap showing number of modalities
unique_nviews_data = combinations_average['n_views'].to_frame()
sns.heatmap(unique_nviews_data.T, cmap='Reds', linewidths=0.1, linecolor='black', square=True,
            cbar=True, cbar_kws=dict(use_gridspec=True, location="bottom"), ax=ax[2], 
            yticklabels='', xticklabels=unique_nviews_data.index).set(xlabel=None)
ax[2].set_ylabel('Number of \nmodalities', labelpad=30, rotation=0, va='center')

# Fourth plot: heatmap showing number of features
features = [52, 2185, 71, 1419, 192, 385]
feature_sums = {}
for combination in combinations_sorted:
    total = sum(features[i] for i, bit in enumerate(combination) if bit == '1')
    feature_sums[combination] = total
features_df = pd.DataFrame(feature_sums, index=['number of features'])
sns.heatmap(features_df, cmap='Oranges', linewidths=0.1, linecolor='black', cbar=True, 
            cbar_kws=dict(use_gridspec=True, location="bottom", ticks=[123, 2150, 4304]), ax=ax[3], square=True,
            yticklabels='', xticklabels=features_df.columns).set(xlabel=None)
ax[3].set_ylabel('Number of \nfeatures', labelpad=30, rotation=0, va='center')
ax[3].set_xticklabels('')

plt.tight_layout()
plt.savefig('figures/first_bench_figures/best_combination_combined.svg', bbox_inches='tight')
plt.show()

To measure the "goodness" or quality of a data combination, a new metric (*__combined metric__*) was created. This metric is the mean of the _silhouette score_, which measures how well-defined and distinct the clusters are, and _adjusted mutual information (AMI) score_, which measures the stability of the clusters. In theory, the best combination of omic views would have the highest combined metric value. 

The first plot is a boxplot showing the combined metric for each combination accross all algorithms and clustering option, with the mean for that option being represented by the blue marker. The heatmap below it shows the number of views present in each combination, and the binary heatmap under shows whether a data type is present in that combination (blue) or if it is not present (white). From the binary heatmap, we can see that the combination corresponding to CNA and miRNA has the overall highest mean value for the combined metric, hence we take CNA and methylation as the best data combination. 

### Raincloud plots

In [ ]:
# Function to plot boxplots for a specific variable
def raincloud_plots_variables(df, column_name, metric_name, ax):
    mean_values = df.groupby(column_name)[metric_name].mean().sort_values(ascending=False)
    sorted_categories = mean_values.index.tolist()
    metric_subsets = [df[df[column_name] == cat][metric_name].values for cat in sorted_categories]
    pt.RainCloud(x=column_name, y=metric_name, data=df, bw=0.2, palette=["tab:blue"],
                 width_viol=0.4, ax=ax, orient="v", move=0.2, order=sorted_categories, alpha=0.8)
    means = df.groupby(column_name)[metric_name].mean().loc[sorted_categories]
    sns.scatterplot(x=range(len(sorted_categories)), y=means.values, ax=ax, color='tab:green', s=100, marker='^', zorder=3)
    ax.set_xticks(range(len(sorted_categories)))
    ax.set_xticklabels(sorted_categories)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(axis='y', alpha=0.4)
    ax.set_axisbelow(True)
    pvalue = kruskal(*metric_subsets).pvalue
    if pvalue >= 0.001:
        pvalue_text = f"p = {pvalue:.3f}"
    else:
        pvalue_text = f"p = {pvalue:.2e}"
    return pvalue, pvalue_text

In [ ]:
normalised_silhouette1_alg = add_normalised_metric(results_clin1, variable_to_normalise='algorithm', metric='silhouette', greater_is_better=True)
stability_metrics_results1_alg = calculate_stability_metrics(normalised_silhouette1_alg, random_state=42, progress_bar=True)
normalised_ami1_alg = add_normalised_metric(stability_metrics_results1_alg, variable_to_normalise='algorithm', metric='AMI', greater_is_better=True)
results1_alg = normalised_ami1_alg.copy()
results1_alg['n_views'] = results1_alg['view_combination'].str.count('1')
results1_alg['combined_metric'] = results1_alg[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results1_alg.sort_values('combined_metric', ascending=False, inplace=True)
results1_alg

In [ ]:
normalised_silhouette1_clusters = add_normalised_metric(results_clin1, variable_to_normalise='n_clusters', metric='silhouette', greater_is_better=True)
stability_metrics_results1_clusters = calculate_stability_metrics(normalised_silhouette1_clusters, random_state=42, progress_bar=True)
normalised_ami1_clusters = add_normalised_metric(stability_metrics_results1_clusters, variable_to_normalise='n_clusters', metric='AMI', greater_is_better=True)
results1_clusters = normalised_ami1_clusters.copy()
results1_clusters['n_views'] = results1_clusters['view_combination'].str.count('1')
results1_clusters['combined_metric'] = results1_clusters[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results1_clusters.sort_values('combined_metric', ascending=False, inplace=True)
results1_clusters

In [ ]:
metric='combined_metric'
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], width_ratios=[0.3, 0.7])

# plot for combinations containing each data type
ax1 = fig.add_subplot(gs[0, :])
plot_data = []
views = ['CNA', 'Methyl', 'Mutation', 'RNAseq', 'RPPA', 'miRNA']
for i, view in enumerate(views):
    subset = results1[results1['view_combination'].str[i] == '1']
    subset['data_type'] = view
    plot_data.append(subset)
combined_df = pd.concat(plot_data)
pvalue, pvalue_text = raincloud_plots_variables(combined_df, 'data_type', metric, ax1)
ax1.set_xlabel('Data type')
ax1.set_ylabel('AMI + silhouette score')
ax1.text(0, 0, pvalue_text)

# plot for algorithm
ax2 = fig.add_subplot(gs[1, :])
pvalue, pvalue_text = raincloud_plots_variables(results1, 'algorithm', metric, ax=ax2)
ax2.set_xlabel('Algorithm')
ax2.set_ylabel('AMI + silhouette score')
ax2.text(0, 0, pvalue_text)

# plot for number of clusters
ax3 = fig.add_subplot(gs[2, 0])
pvalue, pvalue_text = raincloud_plots_variables(results1_clusters, 'n_clusters', metric, ax=ax3)
ax3.set_xlabel('Number of clusters')
ax3.set_ylabel('AMI + silhouette score')
ax3.text(0, 0, pvalue_text)

# plot for combinations with a specific number of views
ax4 = fig.add_subplot(gs[2, 1])
pvalue, pvalue_text = raincloud_plots_variables(results1_alg, 'n_views', metric, ax4)
ax4.set_xlabel('Number of modalities')
ax4.set_ylabel('AMI + silhouette score')
ax4.text(0, 0, pvalue_text)

fig.subplots_adjust(wspace = 0.3, hspace = 0.3)
plt.savefig('figures/first_bench_figures/raincloud_plots_combined.svg', bbox_inches='tight')
plt.show()

The plots above show the distribution for several hyperparameters: data type, algorithm, number of clusters and number of views used in each experiment. Kruskal-Wallis test was performed on each of them to see if these parameters tested were statistically different between the options within each group. There are statistically significant differences in the data type present ($p = 9.02e-05$), algorithm ($p = 1.23e-113$) and number of clusters ($p = 3.54e-0.5$). Since there is a difference in the number of clusters, in future experiments and benchmarks, only n = 2 clusters will be used. Having 2 clusters typically simplifies analysis by allowing direct analysis in a binary fashion. Having more than 2 clusters can introduce complexity, since soe clusters may show overlapping characteristics and make interpretation harder. 

### Scatter plots

In [ ]:
# Function to create new df for data types used
views = ['CNA', 'Methyl', 'Mutation', 'RNAseq', 'RPPA', 'miRNA']
def expand_views(df):
    expanded_rows = []
    for idx, row in df.iterrows():
        view_combination = row['view_combination']
        for i, bit in enumerate(view_combination):
            if bit == '1':
                new_row = row.copy()
                new_row['views_present'] = views[i]
                expanded_rows.append(new_row)
    expanded_df = pd.DataFrame(expanded_rows)
    return expanded_df

# Scatter plots (mean number of clinically enriched parameters)
def plot_scatterplot(df, column_name, ax):
    summarised_by_variable = df.groupby(column_name, as_index=False).mean(numeric_only=True)
    sns.scatterplot(x='pvalue_logrank', y='n_enriched_clinical', data = summarised_by_variable,
                    hue=column_name, style=column_name, palette = "colorblind", markers=True, s=400, ax=ax)
    ax.set_ylim(-0.1, 1)
    ax.grid(alpha=0.4)
    ax.set_ylabel('Mean no. significant clinical parameters')
    ax.set_xlabel('Log-rank test p-value')
    
fig, ax = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
fig.subplots_adjust(wspace = 0.2, hspace = 0.2)
results1_alg_exp = expand_views(results1_alg)
plot_scatterplot(results1_alg_exp, 'algorithm', ax[0])
ax[0].legend(title='Algorithm')
results1_clusters_exp = expand_views(results1_clusters)
plot_scatterplot(results1_clusters_exp, 'n_clusters', ax[1])
ax[1].legend(title='Number\nof clusters')
results1_exp = expand_views(results1)
plot_scatterplot(results1_exp, 'n_views', ax[2])
ax[2].legend(title='Number of \nmodalities')
plot_scatterplot(results1_exp, 'views_present', ax[3])
ax[3].legend(title='Modalities present')

fig.subplots_adjust(wspace = 0.1)
plt.savefig('figures/first_bench_figures/clinical_enrichment.svg', bbox_inches='tight')
plt.show()

To test the enrichment of clinical labels and test whether the clusters formed in each experiment were clinically relevant, enrichment of clinical labels was performed. 22 parameters were tested, related to specific information about the tumor (metastases, cancer in lymph nodes, etc.) and also the patient's personal clinical information (age, history of diabetes, etc.). Kruskal-Wallis test was performed on numeric parameters, and chi-square ($\chi^2$) test was performed on discrete parameters. The p-values for clinical labels were corrected for multiple hypotheses using the Benjamini-Hochberg procedure, at a significance level of 0.05. The mean number of significant clinical parameters was calculated per algorithm, number of clusters, number of views and views present. Significant differences in survival were also calculated using the logrank test at the same significance level. These differences were plotted within each group of hyperparameters previously mentioned. 

From the plots above, we can see that for each hyperparameter, there are no significantly enriched clinical parameters and no statistically significant differences in survival.